In [ ]:
# All the imports
import contextlib
import io
import os
import shutil
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import kurtosis, norm

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.fields import (compute_radial_spectrum, current_density, partial_x_2d, partial_x_fft, partial_y_2d,
                           partial_y_fft, summed_spectrum)
from shared.filtering import iterative_filtering1DWithKCutOff
from shared.loading import grid_spacing, load, output_times
from shared.parallel import parallel_map
from shared.paths import cache_dir, results_dir, run_dir
from shared.plotting import LINE_COLOURS, add_slope_guide, colour_limits, save_figure, show_field, use_style

use_style()
RESULTS = results_dir("apj2015", "ApJ2015")

In [ ]:
# ---- the run ----
BETA_P, BETA_E, BRMS = 1, 1, 0.24
RUN = run_dir(BETA_P, BETA_E, BRMS)            # NAS/DATA/Brms_0.24/beta_p_1/beta_e_1
DX, DY = grid_spacing(RUN)                     # box length / number of cells (0.0625 d_p for this run)
TIMES = output_times(RUN)                      # every output time present in the run folder
FIRST_TIME, LAST_TIME = TIMES[0], TIMES[-1]

# ---- normalisation: B_0 = n_0 = 1, temperatures in units of m_p v_A^2 ----
GUIDE_FIELD = 1.0                              # B_0
T_P0 = BETA_P / 2                              # initial proton temperature (beta_p = 2 n T_p / B_0^2)
T_E = BETA_E / 2                               # electron temperature (isothermal electrons)

# ---- which snapshots to look at ----
SNAPSHOT_TIME = 200                            # maps, increments and spectra
B_PERP_MAP_TIME = 253                          # the |B_perp|^2 map
J_MAGNITUDE_MAP_TIME = 40                      # the |J_par| map
J_SIGNED_MAP_TIME = 42                         # the J_par map
COMPENSATED_SPECTRA_TIMES = [time for time in TIMES if 200 <= time < 300][::20]

# ---- increments of B_y: separations in d_p ----
PDF_SEPARATIONS = [0.75, 3.125, 15.75, 37.5]
KURTOSIS_SEPARATIONS = [0.5, 4.0, 16.0, 64.0, 128.0, 256.0]

# ---- spectra ----
NYQUIST_K = np.pi / DX                         # largest k_perp d_p on this grid
COMPENSATED_K_RANGE = (5e-2, 6)                # k_perp d_p range shown in the compensated spectra
STACK_FACTOR = 10                              # vertical offset between successive compensated spectra
INCLUDE_E_PERP_SPECTRUM = False                # also show the E_perp spectrum (needs Ex, Ey output files)

# ---- movies ----
DIV_B_MOVIE_FPS, B_PERP_MOVIE_FPS = 1, 10

# ---- filtering of B and u: Fast Iterative Filtering with a k cutoff, as in the Paper 1 pipeline ----
FILTER_B = True                                # maps, B_y increments, time series and |B_perp|^2 movie use the filtered B
FILTER_U = True                                # maps, omega_par and the time series use the filtered velocity
K_CUTOFF = 14.12 / 3                           # k_perp d_p above which the filter removes B and u (the Paper 1 value, beta_p = 1)
XI = 1.6                                       # fifpy Xi
# The spectra and div B always use the raw fields: the filter removes the power above K_CUTOFF by design,
# and filtering the components one at a time would add a divergence of its own.

# ---- loops over output times run in parallel processes ----
N_WORKERS = os.cpu_count()                     # each process holds ~20 full fields (~3 GB at 4096^2): lower it if memory runs short
THREADS_PER_WORKER = 1                         # numpy/FFT threads inside each process


def filtered(field, enabled=True):
    """field with the wavenumbers above K_CUTOFF removed, when enabled (fifpy's progress messages hidden)."""
    if not enabled:
        return field
    with contextlib.redirect_stdout(io.StringIO()):
        return iterative_filtering1DWithKCutOff(field, Xi=XI, k=K_CUTOFF, dx=DX)


def load_B(time, components=("Bx", "By", "Bz")):
    """B components at one output time, filtered when FILTER_B is True."""
    return [filtered(load(component, time, RUN), FILTER_B) for component in components]


def load_u(time, components=("Ux1", "Uy1", "Uz1"), density=None):
    """Velocity components (Ux1 / Dn1, ...) at one output time, filtered when FILTER_U is True."""
    density = load("Dn1", time, RUN) if density is None else density
    return [filtered(load(component, time, RUN) / density, FILTER_U) for component in components]

In [ ]:
# One parallel pass over every output time: the time series, and the frames of the two movies
# (div B from the raw B, |B_perp|^2 from the filtered B), so each field is read and filtered only once
SERIES_NAMES = ["J_par_rms", "omega_par_rms", "B_perp_rms", "B_par_rms", "u_perp_rms", "u_par_rms",
                "T_perp_mean", "T_par_mean", "anisotropy_mean", "sigma_c_mean", "sigma_r_mean"]
MOVIES = {"div_B": DIV_B_MOVIE_FPS, "B_perp_squared": B_PERP_MOVIE_FPS}      # movie -> frames per second
FRAME_FOLDERS = {name: cache_dir("apj2015", "ApJ2015", f"{name}_frames") for name in MOVIES}
for folder in FRAME_FOLDERS.values():               # start from empty folders, so no old frames end up in a movie
    shutil.rmtree(folder)
    folder.mkdir(parents=True)

# fixed colour ranges for the movies, taken from SNAPSHOT_TIME
Bx_raw, By_raw = load("Bx", SNAPSHOT_TIME, RUN), load("By", SNAPSHOT_TIME, RUN)
DIV_B_LIMITS = colour_limits(partial_x_fft(Bx_raw, DX) + partial_y_fft(By_raw, DY), centre=0)
B_SNAPSHOT = load_B(SNAPSHOT_TIME)                  # filtered B at SNAPSHOT_TIME; the maps below use it too
B_PERP_SQUARED_LIMITS = colour_limits(B_SNAPSHOT[0]**2 + B_SNAPSHOT[1]**2)
U_SNAPSHOT = load_u(SNAPSHOT_TIME, ("Ux1", "Uy1"))  # filtered u_perp at SNAPSHOT_TIME, for the maps below


def save_frame(field, movie, frame_index, title, centre, limits):
    """One movie frame as a PNG, always the same size in pixels so the frames line up."""
    fig, ax = plt.subplots(figsize=(8, 6))
    show_field(ax, field, DX, DY, centre=centre, limits=limits)
    ax.set_title(title)
    with plt.rc_context({"savefig.bbox": "standard"}):
        fig.savefig(FRAME_FOLDERS[movie] / f"frame_{frame_index:04d}.png")
    plt.close(fig)


def analyse_time(frame_index):
    """The time-series values at TIMES[frame_index]; also draws that time's two movie frames."""
    time = TIMES[frame_index]
    Bx_raw, By_raw, Bz_raw = (load(component, time, RUN) for component in ("Bx", "By", "Bz"))
    save_frame(partial_x_fft(Bx_raw, DX) + partial_y_fft(By_raw, DY), "div_B", frame_index,
               rf'$\nabla \cdot B$, t = {time}', 0, DIV_B_LIMITS)
    Bx, By, Bz = (filtered(component, FILTER_B) for component in (Bx_raw, By_raw, Bz_raw))
    del Bx_raw, By_raw, Bz_raw
    save_frame(Bx**2 + By**2, "B_perp_squared", frame_index, rf'$|B_\perp|^2$, t = {time}', None, B_PERP_SQUARED_LIMITS)

    density = load("Dn1", time, RUN)
    ux, uy, uz = load_u(time, density=density)
    T_perp = load("Pper1", time, RUN) / density
    T_par = load("Ppar1", time, RUN) / density

    J_par = partial_x_fft(By, DX) - partial_y_fft(Bx, DY)
    omega_par = partial_x_fft(uy, DX) - partial_y_fft(ux, DY)
    energy = ux**2 + uy**2 + uz**2 + Bx**2 + By**2 + Bz**2
    return {
        "J_par_rms": np.sqrt(np.mean(J_par**2)),
        "omega_par_rms": np.sqrt(np.mean(omega_par**2)),
        "B_perp_rms": np.sqrt(np.mean(Bx**2 + By**2)),
        "B_par_rms": np.sqrt(np.mean((Bz - GUIDE_FIELD)**2)),
        "u_perp_rms": np.sqrt(np.mean(ux**2 + uy**2)),
        "u_par_rms": np.sqrt(np.mean(uz**2)),
        "T_perp_mean": np.mean(T_perp / T_P0),
        "T_par_mean": np.mean(T_par / T_P0),
        "anisotropy_mean": np.mean(T_perp / T_par),
        "sigma_c_mean": np.mean(2 * (ux*Bx + uy*By + uz*Bz) / energy),
        "sigma_r_mean": np.mean((ux**2 + uy**2 + uz**2 - Bx**2 - By**2 - Bz**2) / energy),
    }


values = parallel_map(analyse_time, range(len(TIMES)), workers=N_WORKERS, threads_per_worker=THREADS_PER_WORKER)
series = {name: np.array([value[name] for value in values]) for name in SERIES_NAMES}
np.savetxt(RESULTS / "time_series.csv", np.column_stack([TIMES] + [series[name] for name in SERIES_NAMES]),
           delimiter=",", header="time," + ",".join(SERIES_NAMES), comments="")

In [ ]:
# J_par at the first output time
Bx_first, By_first, Bz_first = load_B(FIRST_TIME)      # filtered when FILTER_B is True; |delta B| below uses them too
J_par = partial_x_fft(By_first, DX) - partial_y_fft(Bx_first, DY)

fig, ax = plt.subplots(figsize=(10, 8))
show_field(ax, J_par, DX, DY, centre=0)
ax.set_title(rf"$J_\parallel$, t = {FIRST_TIME}")
save_figure(fig, RESULTS, f"J_par_t{FIRST_TIME}")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(TIMES, series["J_par_rms"], linewidth=1.5, label=r'$J^{\mathrm{rms}}_{\parallel}$')
ax.plot(TIMES, series["omega_par_rms"], linewidth=1.5, label=r'$\omega^{\mathrm{rms}}_{\parallel}$')
ax.set_xlabel(r'Time index (or $t\, [\Omega_p^{-1}]$)')
ax.set_ylabel('RMS Value')
ax.legend()
save_figure(fig, RESULTS, "rms_J_par_and_omega_par_vs_time")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
# magnetic field: first line colour, velocity: second
ax.plot(TIMES, series["B_perp_rms"], color=LINE_COLOURS[0], linestyle='-', linewidth=1.5, label=r'$B^{\mathrm{rms}}_\perp$')
ax.plot(TIMES, series["B_par_rms"], color=LINE_COLOURS[0], linestyle='--', linewidth=1.5, label=r'$B^{\mathrm{rms}}_\parallel$')
ax.plot(TIMES, series["u_perp_rms"], color=LINE_COLOURS[1], linestyle='-', linewidth=1.5, label=r'$u^{\mathrm{rms}}_\perp$')
ax.plot(TIMES, series["u_par_rms"], color=LINE_COLOURS[1], linestyle='--', linewidth=1.5, label=r'$u^{\mathrm{rms}}_\parallel$')
ax.set_xlabel(r'Time index (or $t\, [\Omega_p^{-1}]$)')
ax.set_ylabel('RMS Value')
ax.legend()
save_figure(fig, RESULTS, "rms_B_and_u_vs_time")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(TIMES, series["T_perp_mean"], linestyle='--', linewidth=1.5, label=r'$\langle T_{p\perp}/T_{0}\rangle$')
ax.plot(TIMES, series["T_par_mean"], linestyle='--', linewidth=1.5, label=r'$\langle T_{p\parallel}/T_{0}\rangle$')
ax.plot(TIMES, series["anisotropy_mean"], linestyle='-', linewidth=1.5, label=r'$\langle A_{p}\rangle$')
ax.set_xlabel(r'Time index (or $t\, [\Omega_p^{-1}]$)')
ax.set_ylabel('Mean value')
ax.legend()
save_figure(fig, RESULTS, "mean_temperatures_and_anisotropy_vs_time")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(TIMES, series["sigma_c_mean"], linestyle='--', linewidth=1.5, label=r'$\langle \sigma_{C} \rangle$')
ax.plot(TIMES, series["sigma_r_mean"], linestyle='--', linewidth=1.5, label=r'$\langle \sigma_{R} \rangle$')
ax.set_xlabel(r'Time index (or $t\, [\Omega_p^{-1}]$)')
ax.set_ylabel('Mean value')
ax.set_ylim(-1, 1)                             # the full range of sigma_C and sigma_R
ax.legend()
save_figure(fig, RESULTS, "mean_cross_and_residual_helicity_vs_time")
plt.show()

In [ ]:
# |u| and |delta B| at the first output time
ux, uy, uz = load_u(FIRST_TIME)                    # filtered when FILTER_U is True
u_magnitude = np.sqrt(ux**2 + uy**2 + uz**2)
delta_B_magnitude = np.sqrt(Bx_first**2 + By_first**2 + (Bz_first - GUIDE_FIELD)**2)

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
show_field(axs[0], u_magnitude, DX, DY)
axs[0].set_title(rf'$|u|$, t = {FIRST_TIME}')
show_field(axs[1], delta_B_magnitude, DX, DY)
axs[1].set_title(rf'$|\delta B|$, t = {FIRST_TIME}')
plt.tight_layout()
save_figure(fig, RESULTS, f"u_and_delta_B_magnitude_t{FIRST_TIME}")
plt.show()

In [ ]:
# |B_perp|^2
Bx, By = load_B(B_PERP_MAP_TIME, ("Bx", "By"))
fig, ax = plt.subplots(figsize=(10, 8))
show_field(ax, Bx**2 + By**2, DX, DY)
ax.set_title(rf'$|B_\perp|^2$, t = {B_PERP_MAP_TIME}')
save_figure(fig, RESULTS, f"B_perp_squared_t{B_PERP_MAP_TIME}")
plt.show()

In [ ]:
# |u_perp|^2
ux, uy = U_SNAPSHOT                                 # filtered u_perp at SNAPSHOT_TIME, from the time-series pass
fig, ax = plt.subplots(figsize=(10, 8))
show_field(ax, ux**2 + uy**2, DX, DY)
ax.set_title(rf'$|u_\perp|^2$, t = {SNAPSHOT_TIME}')
save_figure(fig, RESULTS, f"u_perp_squared_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
# |B_perp|^2 as a bare image (no colour bar or labels)
Bx, By, _ = B_SNAPSHOT                              # filtered B at SNAPSHOT_TIME, from the time-series pass
fig, ax = plt.subplots(figsize=(10, 8))
show_field(ax, Bx**2 + By**2, DX, DY, colorbar=False, axis_labels=False)
save_figure(fig, RESULTS, f"B_perp_squared_t{SNAPSHOT_TIME}_image_only")
plt.show()

In [ ]:
# |J_par|
Bx, By = load_B(J_MAGNITUDE_MAP_TIME, ("Bx", "By"))
J_par_magnitude = np.abs(partial_x_fft(By, DX) - partial_y_fft(Bx, DY))
fig, ax = plt.subplots(figsize=(10, 8))
show_field(ax, J_par_magnitude, DX, DY)
ax.set_title(rf'$|J_\parallel|$, t = {J_MAGNITUDE_MAP_TIME}')
save_figure(fig, RESULTS, f"J_par_magnitude_t{J_MAGNITUDE_MAP_TIME}")
plt.show()

In [ ]:
# omega_par (spectral derivatives; the finite-difference version is kept for comparison)
ux, uy = U_SNAPSHOT                                 # filtered u_perp at SNAPSHOT_TIME, from the time-series pass
omega_par_finite_difference = partial_x_2d(uy, DX) - partial_y_2d(ux, DY)
omega_par = partial_x_fft(uy, DX) - partial_y_fft(ux, DY)

fig, ax = plt.subplots(figsize=(10, 8))
show_field(ax, omega_par, DX, DY, centre=0)
ax.set_title(rf'$\omega_\parallel$, t = {SNAPSHOT_TIME}')
save_figure(fig, RESULTS, f"omega_par_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
# Delta T_p / T_0
density = load("Dn1", SNAPSHOT_TIME, RUN)
T_perp = load("Pper1", SNAPSHOT_TIME, RUN) / density
T_par = load("Ppar1", SNAPSHOT_TIME, RUN) / density
T_total = (2 * T_perp + T_par) / 3
delta_T_over_T0 = (T_total - T_P0) / T_P0

fig, ax = plt.subplots(figsize=(10, 8))
show_field(ax, delta_T_over_T0, DX, DY, centre=0)
ax.set_title(rf'$\Delta T_{{p}}/T_{{0}}$, t = {SNAPSHOT_TIME}')
save_figure(fig, RESULTS, f"delta_T_over_T0_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
# A_p = T_perp / T_par next to omega_par
anisotropy = T_perp / T_par

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
show_field(axs[0], anisotropy, DX, DY, centre=1)
axs[0].set_title(rf'$A_p$, t = {SNAPSHOT_TIME}')
show_field(axs[1], omega_par, DX, DY, centre=0)
axs[1].set_title(rf'$\omega_\parallel$, t = {SNAPSHOT_TIME}')
plt.tight_layout()
save_figure(fig, RESULTS, f"anisotropy_and_omega_par_t{SNAPSHOT_TIME}")
plt.show()

All four fields at `SNAPSHOT_TIME` in one figure (|B_perp|^2, J_par, Delta T_p/T_0, A_p).
This came from the separate notebook `ApJ2015-Copy1` (now in `archive/apj2015/`).

In [ ]:
Bx, By, _ = B_SNAPSHOT                              # filtered B at SNAPSHOT_TIME, from the time-series pass
J_par = partial_x_fft(By, DX) - partial_y_fft(Bx, DY)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
panels = [(Bx**2 + By**2, None, r'$|B_\perp|^2$'), (J_par, 0, r'$J_\parallel$'),
          (delta_T_over_T0, 0, r'$\Delta T_{p}/T_{0}$'), (anisotropy, 1, r'$A_{p}$')]
for ax, (field, centre, title) in zip(axes.ravel(), panels):
    show_field(ax, field, DX, DY, centre=centre)
    ax.set_title(title)
plt.tight_layout()
save_figure(fig, RESULTS, f"four_fields_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
# PDFs of the B_y increments at several separations
By = B_SNAPSHOT[1]                                  # filtered B_y at SNAPSHOT_TIME, from the time-series pass
fig, ax = plt.subplots(figsize=(8, 6))
for separation, colour in zip(PDF_SEPARATIONS, LINE_COLOURS):
    cells = int(round(separation / DX))
    if not 1 <= cells < By.shape[0]:
        continue
    increments = (By[cells:, :] - By[:-cells, :]).ravel()
    increments /= np.std(increments)
    density_values, bin_edges = np.histogram(increments, bins=200, density=True)
    bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    ax.semilogy(bin_centres, gaussian_filter1d(density_values, sigma=2), '-', color=colour,
                label=rf'$\ell = {separation:g}\,d_p$')

normalised_increment = np.linspace(-5, 5, 500)
ax.semilogy(normalised_increment, norm.pdf(normalised_increment), 'k--', label='Gaussian')
ax.set_xlabel(r'$\Delta B_y / \sigma$', fontsize=14)
ax.set_ylabel('PDF', fontsize=14)
ax.set_title(rf'PDFs of $\Delta B_y$ at different scales, t = {SNAPSHOT_TIME}', fontsize=15)
ax.set_ylim(1e-3, None)
ax.set_xlim(-6, 6)
ax.legend(fontsize=12)
save_figure(fig, RESULTS, f"increment_pdfs_By_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
# Excess kurtosis of the B_y increments against k = 2 pi / ell
separations_used, excess_kurtosis = [], []
for separation in KURTOSIS_SEPARATIONS:
    cells = int(round(separation / DX))
    if not 1 <= cells < By.shape[0]:
        continue                                # an increment over the whole box is identically zero
    increments = (By[cells:, :] - By[:-cells, :]).ravel()
    increments /= np.std(increments)
    separations_used.append(cells * DX)
    excess_kurtosis.append(kurtosis(increments, fisher=False) - 3)

wavenumbers = 2 * np.pi / np.array(separations_used)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(wavenumbers, excess_kurtosis, color=LINE_COLOURS[4])
ax.set_xscale('log')
ax.set_xlabel(r'$k\,d_p = 2\pi d_p/\ell$', fontsize=14)
ax.set_ylabel('Excess Kurtosis', fontsize=14)
ax.set_title(rf'Excess kurtosis of $\Delta B_y$ against wavenumber, t = {SNAPSHOT_TIME}', fontsize=15)
ax.axhline(0, color='k', linestyle='--', linewidth=1)
save_figure(fig, RESULTS, f"excess_kurtosis_By_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
# Spectra of B_perp and u_perp at SNAPSHOT_TIME, and u_perp at the first output (the noise level).
# Spectra use the raw B: the filter removes the power above K_CUTOFF by design.
Bx, By = load("Bx", SNAPSHOT_TIME, RUN), load("By", SNAPSHOT_TIME, RUN)
k_perp, B_perp_spectrum = summed_spectrum((Bx, By), DX, DY)
density = load("Dn1", SNAPSHOT_TIME, RUN)
_, u_perp_spectrum = summed_spectrum((load("Ux1", SNAPSHOT_TIME, RUN) / density, load("Uy1", SNAPSHOT_TIME, RUN) / density), DX, DY)
density = load("Dn1", FIRST_TIME, RUN)
_, noise_spectrum = summed_spectrum((load("Ux1", FIRST_TIME, RUN) / density, load("Uy1", FIRST_TIME, RUN) / density), DX, DY)

In [ ]:
NOISE_K_MIN = 6.1                               # show the noise spectrum above this k_perp d_p
SLOPE_GUIDES = [(-5/3, 0.12, 1.96, r'$k_\perp^{-5/3}$'), (-3/2, 0.12, 1.2, r'$k_\perp^{-3/2}$'),
                (-3, 2.9, 14.7, r'$k_\perp^{-3}$')]        # (slope, k_min, k_max, label)

shown = (k_perp > 0) & (k_perp <= NYQUIST_K)
fig, ax = plt.subplots(figsize=(8, 6))
ax.loglog(k_perp[shown], B_perp_spectrum[shown], label=r'$P_{B_\perp}$', linewidth=2)
ax.loglog(k_perp[shown], u_perp_spectrum[shown], label=r'$P_{u_\perp}$', linewidth=2)
noisy = shown & (k_perp >= NOISE_K_MIN)
ax.loglog(k_perp[noisy], noise_spectrum[noisy], label='noise')
for (slope, k_min, k_max, label), spectrum in zip(SLOPE_GUIDES, [B_perp_spectrum, u_perp_spectrum, B_perp_spectrum]):
    add_slope_guide(ax, k_perp, spectrum, slope, k_min, k_max, label=label)
ax.set_xlabel(r'$k_\perp d_{p}$')
ax.set_ylabel(r'$P(k_\perp)$')
ax.legend(loc='lower left')
save_figure(fig, RESULTS, f"spectra_B_perp_u_perp_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
# Compensated spectra over time: B_perp * k^{5/3}, u_perp * k^{3/2} (raw B; the times are computed in parallel)
def perpendicular_spectra_at(time):
    """(k_perp, B_perp spectrum, u_perp spectrum) at one output time."""
    k_perp, B_perp_spectrum = summed_spectrum((load("Bx", time, RUN), load("By", time, RUN)), DX, DY)
    density = load("Dn1", time, RUN)
    _, u_perp_spectrum = summed_spectrum((load("Ux1", time, RUN) / density, load("Uy1", time, RUN) / density), DX, DY)
    return k_perp, B_perp_spectrum, u_perp_spectrum


spectra = parallel_map(perpendicular_spectra_at, COMPENSATED_SPECTRA_TIMES, workers=N_WORKERS,
                       threads_per_worker=THREADS_PER_WORKER)
k_perp = spectra[0][0]
B_perp_spectra = [B_perp_spectrum for _, B_perp_spectrum, _ in spectra]
u_perp_spectra = [u_perp_spectrum for _, _, u_perp_spectrum in spectra]

In [ ]:
def plot_compensated(spectra, exponent, exponent_label, inertial_range, figure_name):
    """Stack k^exponent * P(k) for each time; the dashed line marks each curve's mean over inertial_range."""
    shown = (k_perp >= COMPENSATED_K_RANGE[0]) & (k_perp <= COMPENSATED_K_RANGE[1])
    inertial = (k_perp >= inertial_range[0]) & (k_perp <= inertial_range[1])
    fig, ax = plt.subplots(figsize=(8, 6))
    for position, (time, spectrum) in enumerate(zip(COMPENSATED_SPECTRA_TIMES, spectra)):
        compensated = spectrum * k_perp**exponent * STACK_FACTOR**position
        ax.loglog(k_perp[shown], compensated[shown], linewidth=2, label=f"t = {time}")
        ax.axhline(np.mean(compensated[inertial]), color='k', linestyle='--', linewidth=1)
    ax.set_xlabel(r'$k_\perp d_{p}$')
    ax.set_ylabel(rf'$k_\perp^{{{exponent_label}}}\,P(k_\perp)$ (shifted)')
    ax.legend(loc='lower left')
    save_figure(fig, RESULTS, figure_name)
    plt.show()

plot_compensated(B_perp_spectra, 5/3, '5/3', (0.12, 1.96), "compensated_spectra_B_perp")
plot_compensated(u_perp_spectra, 3/2, '3/2', (0.12, 1.2), "compensated_spectra_u_perp")

In [ ]:
# Spectra of J, n, |B| and B_par at SNAPSHOT_TIME (raw B: the filter removes the power above K_CUTOFF by design)
Bx, By, Bz = (load(component, SNAPSHOT_TIME, RUN) for component in ("Bx", "By", "Bz"))
density = load("Dn1", SNAPSHOT_TIME, RUN)
Jx, Jy, Jz = current_density(Bx, By, Bz, DX, DY)

k_perp, J_spectrum = summed_spectrum((Jx, Jy, Jz), DX, DY)
density_spectrum = compute_radial_spectrum(density, DX, DY).sp
B_magnitude_spectrum = compute_radial_spectrum(np.sqrt(Bx**2 + By**2 + Bz**2), DX, DY).sp
B_par_spectrum = compute_radial_spectrum(Bz, DX, DY).sp

SLOPE_GUIDES = [(1/3, 0.245, 2.2, r'$k_\perp^{1/3}$', J_spectrum), (-1, 2.95, 12.3, r'$k_\perp^{-1}$', J_spectrum),
                (-2.8, 2.95, 14.7, r'$k_\perp^{-2.8}$', density_spectrum)]

shown = (k_perp > 0) & (k_perp <= NYQUIST_K)
fig, ax = plt.subplots(figsize=(8, 6))
for spectrum, label in [(J_spectrum, r'$P_{J}$'), (density_spectrum, r'$P_{n}$'),
                        (B_magnitude_spectrum, r'$P_{|B|}$'), (B_par_spectrum, r'$P_{B_\parallel}$')]:
    ax.loglog(k_perp[shown], spectrum[shown], linewidth=2, label=label)
for slope, k_min, k_max, label, spectrum in SLOPE_GUIDES:
    add_slope_guide(ax, k_perp, spectrum, slope, k_min, k_max, label=label)
ax.set_xlabel(r'$k_\perp d_{p}$')
ax.set_ylabel(r'$P(k_\perp)$')
ax.legend(loc='lower left')
save_figure(fig, RESULTS, f"spectra_J_n_B_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
# Spectra of the terms of the generalised Ohm's law: u x B, J x B and the electron pressure gradient
density = load("Dn1", SNAPSHOT_TIME, RUN)
ux, uy, uz = (load(component, SNAPSHOT_TIME, RUN) / density for component in ("Ux1", "Uy1", "Uz1"))

u_cross_B = (uy*Bz - uz*By, ux*Bz - Bx*uz, ux*By - Bx*uy)
J_cross_B = (Jy*Bz - Jz*By, Jx*Bz - Bx*Jz, Jx*By - Bx*Jy)
pressure_gradient = (partial_x_fft(density, DX) * T_E, partial_y_fft(density, DY) * T_E)

terms = [(u_cross_B, r'$P_{u \times B}$'), (J_cross_B, r'$P_{J \times B}$'), (pressure_gradient, r'$P_{\nabla p_{e}}$')]
if INCLUDE_E_PERP_SPECTRUM:
    terms.append(((load("Ex", SNAPSHOT_TIME, RUN), load("Ey", SNAPSHOT_TIME, RUN)), r'$P_{E_\perp}$'))

SHADED_K_RANGE = (7, 20)                        # k_perp d_p range shaded in grey
fig, ax = plt.subplots(figsize=(8, 6))
for components, label in terms:
    spectrum = sum(compute_radial_spectrum(component, DX, DY).sp for component in components)
    ax.loglog(k_perp[shown], spectrum[shown], linewidth=2, label=label)
    if label == r'$P_{u \times B}$':
        add_slope_guide(ax, k_perp, spectrum, -0.8, 2.45, 24.5, label=r'$k_\perp^{-0.8}$')
ax.axvspan(*SHADED_K_RANGE, color='lightgray', alpha=0.5)
ax.set_xlabel(r'$k_\perp d_{p}$')
ax.set_ylabel(r'$P(k_\perp)$')
ax.legend(loc='lower left')
save_figure(fig, RESULTS, f"spectra_ohms_law_terms_t{SNAPSHOT_TIME}")
plt.show()

In [ ]:
# Movie of div B over every output time (raw B; colour range from SNAPSHOT_TIME, kept fixed).
# The frames were drawn in the time-series pass; they stay in the cache folder until that pass runs again.
def assemble_movie(movie, movie_path):
    """Join one movie's frames into an mp4 with ffmpeg."""
    subprocess.run([plt.rcParams["animation.ffmpeg_path"], "-y", "-loglevel", "error",
                    "-framerate", str(MOVIES[movie]), "-i", str(FRAME_FOLDERS[movie] / "frame_%04d.png"),
                    "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", "-c:v", "libx264", "-pix_fmt", "yuv420p",
                    "-b:v", "1800k", str(movie_path)], check=True)
    print(f"saved {movie_path}")


def divergence_of_B(time):
    """div B of the raw B at one output time."""
    return partial_x_fft(load("Bx", time, RUN), DX) + partial_y_fft(load("By", time, RUN), DY)


assemble_movie("div_B", RESULTS / "div_B_movie.mp4")

In [ ]:
# Movie of |B_perp|^2 over every output time (filtered B when FILTER_B is True; colour range from SNAPSHOT_TIME)
assemble_movie("B_perp_squared", RESULTS / "B_perp_squared_movie.mp4")

In [ ]:
# div B at the last output time and at SNAPSHOT_TIME
for time in (LAST_TIME, SNAPSHOT_TIME):
    divergence = divergence_of_B(time)
    fig, ax = plt.subplots(figsize=(8, 6))
    show_field(ax, divergence, DX, DY, centre=0, label=r'$\nabla \cdot \mathbf{B}$')
    ax.set_title(rf'Divergence of $\mathbf{{B}}$ at t = {time}')
    save_figure(fig, RESULTS, f"div_B_t{time}")
    plt.show()
    print(f"t = {time}: max |div B| = {np.max(np.abs(divergence)):.3e}")

In [ ]:
print("RMS of B_perp at the first output:", series["B_perp_rms"][0])

In [ ]:
# B_perp rms against time, with its peak marked
peak_index = int(np.argmax(series["B_perp_rms"]))
peak_time, peak_value = TIMES[peak_index], series["B_perp_rms"][peak_index]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(TIMES, series["B_perp_rms"], linewidth=1.5, label=r'$B^{\mathrm{rms}}_{\perp}$')
ax.axvline(x=peak_time, color=LINE_COLOURS[3], linestyle='--', label='Peak')
ax.annotate(f'Peak\n({peak_time}, {peak_value:.2f})', xy=(peak_time, peak_value),
            xytext=(peak_time + 0.02 * (TIMES[-1] - TIMES[0]), peak_value),
            arrowprops=dict(facecolor='black', arrowstyle='->'))
ax.set_xlabel(r'Time index (or $t\, [\Omega_p^{-1}]$)')
ax.set_ylabel('RMS Value')
ax.legend()
save_figure(fig, RESULTS, "rms_B_perp_vs_time_with_peak")
plt.show()

In [ ]:
# J_par
Bx, By = load_B(J_SIGNED_MAP_TIME, ("Bx", "By"))
J_par = partial_x_fft(By, DX) - partial_y_fft(Bx, DY)
fig, ax = plt.subplots(figsize=(10, 8))
show_field(ax, J_par, DX, DY, centre=0)
ax.set_title(rf'$J_\parallel$, t = {J_SIGNED_MAP_TIME}')
save_figure(fig, RESULTS, f"J_par_t{J_SIGNED_MAP_TIME}")
plt.show()